# KPI 6 — Customer Activity

**requirements.md (Customer Activity, KPI 6):** "Orders per customer, average order value, repeat purchase rate."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_6_customer_activity` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **active customer** (`customer_id`) — the 1,071 customers with ≥ 1 valid-date order line (`order_date_valid = true`, KPI 1/2 convention). The 8 customers whose orders all fall on invalid dates are excluded, consistently with the company views. Customer attributes (`gender`, `is_active`, `company_id`) come from the Type-1 `dim_customers`; company name/country from the current SCD2 version of `dim_companies` (`__END_AT IS NULL`).
* **Orders (KPI 5 convention):** `order_count` = distinct valid-date orders of **any** status (cancelled included); `net_order_count` = distinct orders excluding `status = 'cancelled'`. An order never spans customers or companies (verified). 22 customers have all their valid orders cancelled → `net_order_count = 0`, `net_spend = 0.00`, and `avg_order_value_net` is deliberately **NULL** (no fake zero).
* **Spend (KPI 1 bases):** `gross_spend` = `SUM(line_total)` all statuses on valid dates; `net_spend` = same excluding `'cancelled'`. Totals reconcile exactly with KPI 1–5 (net 28,333,498.48 / gross 30,723,269.53 / 10,817 orders).
* **Average order value:** `avg_order_value_gross` = `ROUND(gross_spend / order_count, 2)` and `avg_order_value_net` = `ROUND(net_spend / net_order_count, 2)` — each AOV pairs a numerator and denominator on the **same** status basis (gross–all-statuses, net–excluding-cancelled), so they are self-consistent.
* **Repeat purchase:** `is_repeat_customer` = `order_count >= 2` (any status — the repeat-purchase moment is placing another order, even if it was later cancelled). The repeat **rate** is a rollup over this view: repeat customers ÷ all active customers (799 / 1,071 = 74.6%), computed in the summary cell below.
* **Recency:** `days_since_last_order` = days between the customer's last valid-date order and the latest valid order date in the fact (the data's “as-of” date) — 0 = the most recent buyer.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_6_customer_activity
TBLPROPERTIES (
  'comment' = 'KPI 6 - Customer Activity (requirements.md: orders per customer, average order value, repeat purchase rate). Grain: one row per active customer_id - the 1,071 customers with >= 1 valid-date order line (order_date_valid = true, KPI 1/2 convention; 8 customers whose orders all fall on invalid dates excluded). Customer attributes from Type-1 dim_customers (gender, is_active, company_id); company name/country from current dim_companies version (__END_AT IS NULL). order_count = COUNT(DISTINCT order_id) valid-date orders ANY status (KPI 5 convention); net_order_count = distinct orders excluding status = ''cancelled''; an order never spans customers or companies (verified). gross_spend = SUM(line_total) all statuses valid dates; net_spend = same excluding ''cancelled'' (KPI 1 bases; totals reconcile exactly with KPI 1-5). avg_order_value_gross = ROUND(gross_spend / order_count, 2); avg_order_value_net = ROUND(net_spend / net_order_count, 2) - each AOV pairs numerator and denominator on the same status basis. 22 all-cancelled customers: net_order_count = 0, avg_order_value_net NULL by design (no fake zero). is_repeat_customer = order_count >= 2 (any status); repeat rate is a rollup: repeat customers / active customers = 799/1071 = 74.6%. days_since_last_order = DATEDIFF(latest valid order date in fact, customer''s last valid order date) - 0 = most recent buyer. Aggregation keyed on customer_id (business key) via the fact''s Type-1 customer_sk.'
)
AS
WITH lines AS (
  SELECT
    f.customer_sk,
    f.order_id,
    f.order_date,
    f.status,
    f.line_total
  FROM b2b_ecommerce.gold.fact_order_items f
  WHERE f.order_date_valid = true
),
mapped AS (
  -- business key via the fact's Type-1 customer_sk (1:1 in dim, verified)
  SELECT
    d.customer_id,
    l.order_id,
    l.order_date,
    l.status,
    l.line_total
  FROM lines l
  JOIN b2b_ecommerce.gold.dim_customers d
    ON d.customer_sk = l.customer_sk
),
per_customer AS (
  SELECT
    customer_id,
    MIN(order_date)                                                      AS first_order_date,
    MAX(order_date)                                                      AS last_order_date,
    COUNT(DISTINCT order_id)                                             AS order_count,
    COUNT(DISTINCT CASE WHEN status <> 'cancelled'
                       THEN order_id END)                                 AS net_order_count,
    SUM(line_total)                                                      AS gross_spend,
    SUM(CASE WHEN status <> 'cancelled'
             THEN line_total ELSE 0 END)                                  AS net_spend
  FROM mapped
  GROUP BY customer_id
),
as_of AS (
  -- the fact''s latest valid order date = the data''s as-of date for recency
  SELECT MAX(order_date) AS latest_order_date
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
)
SELECT
  p.customer_id,
  d.gender,
  d.is_active,
  d.company_id,
  c.name                                                AS company_name,
  c.country,
  p.first_order_date,
  p.last_order_date,
  DATEDIFF(a.latest_order_date, p.last_order_date)      AS days_since_last_order,
  p.order_count,
  p.net_order_count,
  (p.order_count >= 2)                                  AS is_repeat_customer,
  p.gross_spend,
  p.net_spend,
  ROUND(p.gross_spend / p.order_count, 2)               AS avg_order_value_gross,
  ROUND(p.net_spend / NULLIF(p.net_order_count, 0), 2)  AS avg_order_value_net
FROM per_customer p
JOIN b2b_ecommerce.gold.dim_customers d
  ON d.customer_id = p.customer_id
JOIN b2b_ecommerce.gold.dim_companies c
  ON c.company_id = d.company_id
 AND c.__END_AT IS NULL
CROSS JOIN as_of a

In [0]:
%sql
-- Verification: shape, totals, rate anchors. Expected: 1,071 rows = 1,071 distinct
-- customers across exactly 390 buying companies (KPI 2/5 reconciliation),
-- SUM(order_count) = 10,817 = total valid-date orders (KPI 5), SUM(net_spend) =
-- 28,333,498.48 and SUM(gross_spend) = 30,723,269.53 (exact KPI 1-5 totals),
-- 799 repeat + 272 single-order = 1,071, 22 all-cancelled customers with NULL
-- net AOV, 0 null customer/company/measure values, days_since_last_order >= 0.
SELECT
  COUNT(*)                                            AS rows_customers,
  COUNT(DISTINCT customer_id)                          AS distinct_customers,
  COUNT(DISTINCT company_id)                           AS distinct_companies,
  SUM(order_count)                                    AS total_orders,
  SUM(net_order_count)                                AS total_net_orders,
  SUM(CASE WHEN is_repeat_customer THEN 1 ELSE 0 END)  AS repeat_customers,
  SUM(CASE WHEN order_count = 1 THEN 1 ELSE 0 END)   AS single_order_customers,
  ROUND(100 * SUM(CASE WHEN is_repeat_customer THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                      AS repeat_rate_pct,
  SUM(CASE WHEN net_order_count = 0 THEN 1 ELSE 0 END) AS all_cancelled_customers,
  SUM(CASE WHEN net_order_count = 0 AND avg_order_value_net IS NOT NULL THEN 1 ELSE 0 END) AS bad_net_aov,
  SUM(gross_spend)                                    AS sum_gross_spend,
  SUM(net_spend)                                      AS sum_net_spend,
  ROUND(SUM(net_spend) / SUM(net_order_count), 2)     AS overall_aov_net,
  ROUND(SUM(gross_spend) / SUM(order_count), 2)        AS overall_aov_gross,
  MIN(days_since_last_order)                          AS min_days_since_last,
  MAX(days_since_last_order)                          AS max_days_since_last,
  SUM(CASE WHEN customer_id IS NULL OR company_id IS NULL OR company_name IS NULL
                OR order_count IS NULL OR gross_spend IS NULL OR net_spend IS NULL
                OR first_order_date IS NULL OR last_order_date IS NULL
                OR days_since_last_order IS NULL THEN 1 ELSE 0 END) AS null_violations
FROM b2b_ecommerce.gold.kpi_6_customer_activity

In [0]:
%sql
-- Verification: independently recompute every customer straight from the raw
-- fact keyed on customer_sk (no dim join for the measures, no shared CTEs with
-- the view), then compare against the view on both order counts, both spend
-- bases, repeat flag, first/last order dates, both AOVs and recency. Expected:
-- customers_checked = 1,071 and every match count = 1,071 (a different path than
-- the view's customer_id aggregation; identical because customer_id <->
-- customer_sk is 1:1 in the Type-1 dim).
WITH recompute AS (
  SELECT
    customer_sk,
    MIN(order_date)                                        AS first_order_date,
    MAX(order_date)                                        AS last_order_date,
    COUNT(DISTINCT order_id)                               AS order_count,
    COUNT(DISTINCT CASE WHEN status <> 'cancelled'
                       THEN order_id END)                   AS net_order_count,
    SUM(line_total)                                        AS gross_spend,
    SUM(CASE WHEN status <> 'cancelled'
             THEN line_total ELSE 0 END)                    AS net_spend
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
  GROUP BY customer_sk
),
as_of AS (
  SELECT MAX(order_date) AS latest_order_date
  FROM b2b_ecommerce.gold.fact_order_items
  WHERE order_date_valid = true
),
joined AS (
  SELECT
    v.customer_id,
    r.order_count                                     AS rc_order_count,
    v.order_count                                     AS v_order_count,
    r.net_order_count                                 AS rc_net_order_count,
    v.net_order_count                                 AS v_net_order_count,
    r.gross_spend                                     AS rc_gross_spend,
    v.gross_spend                                     AS v_gross_spend,
    r.net_spend                                       AS rc_net_spend,
    v.net_spend                                       AS v_net_spend,
    (r.order_count >= 2)                              AS rc_is_repeat,
    v.is_repeat_customer                              AS v_is_repeat,
    r.first_order_date                                AS rc_first,
    v.first_order_date                                AS v_first,
    r.last_order_date                                 AS rc_last,
    v.last_order_date                                 AS v_last,
    ROUND(r.gross_spend / r.order_count, 2)           AS rc_aov_gross,
    v.avg_order_value_gross                           AS v_aov_gross,
    ROUND(r.net_spend / NULLIF(r.net_order_count, 0), 2) AS rc_aov_net,
    v.avg_order_value_net                             AS v_aov_net,
    DATEDIFF(a.latest_order_date, r.last_order_date)  AS rc_days_since,
    v.days_since_last_order                           AS v_days_since
  FROM recompute r
  JOIN b2b_ecommerce.gold.dim_customers d ON d.customer_sk = r.customer_sk
  JOIN b2b_ecommerce.gold.kpi_6_customer_activity v ON v.customer_id = d.customer_id
  CROSS JOIN as_of a
)
SELECT
  COUNT(*)                                              AS customers_checked,
  SUM(CASE WHEN rc_order_count = v_order_count THEN 1 ELSE 0 END)             AS order_count_matches,
  SUM(CASE WHEN rc_net_order_count = v_net_order_count THEN 1 ELSE 0 END)     AS net_order_count_matches,
  SUM(CASE WHEN rc_gross_spend = v_gross_spend THEN 1 ELSE 0 END)             AS gross_spend_matches,
  SUM(CASE WHEN rc_net_spend = v_net_spend THEN 1 ELSE 0 END)                 AS net_spend_matches,
  SUM(CASE WHEN rc_is_repeat = v_is_repeat THEN 1 ELSE 0 END)                 AS repeat_flag_matches,
  SUM(CASE WHEN rc_first = v_first AND rc_last = v_last THEN 1 ELSE 0 END)     AS order_dates_matches,
  SUM(CASE WHEN rc_aov_gross = v_aov_gross THEN 1 ELSE 0 END)                 AS aov_gross_matches,
  SUM(CASE WHEN rc_aov_net <=> v_aov_net THEN 1 ELSE 0 END)                   AS aov_net_matches,
  SUM(CASE WHEN rc_days_since = v_days_since THEN 1 ELSE 0 END)               AS recency_matches
FROM joined

In [0]:
%sql
-- Repeat purchase rate rollup (requirements: "repeat purchase rate"): the
-- overall rate plus the single-order vs repeat split, with weighted AOVs
-- (total spend / total orders - not an average of per-customer averages).
SELECT
  COUNT(*)                                            AS active_customers,
  SUM(CASE WHEN is_repeat_customer THEN 1 ELSE 0 END)  AS repeat_customers,
  ROUND(100 * SUM(CASE WHEN is_repeat_customer THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                      AS repeat_rate_pct,
  SUM(CASE WHEN order_count = 1 THEN 1 ELSE 0 END)   AS single_order_customers,
  ROUND(100 * SUM(CASE WHEN order_count = 1 THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                      AS single_order_pct,
  SUM(order_count)                                    AS total_orders,
  ROUND(AVG(order_count), 2)                          AS avg_orders_per_customer,
  MAX(order_count)                                    AS max_orders_per_customer,
  ROUND(SUM(net_spend) / SUM(net_order_count), 2)     AS overall_aov_net,
  ROUND(SUM(gross_spend) / SUM(order_count), 2)       AS overall_aov_gross,
  ROUND(AVG(days_since_last_order), 1)               AS avg_days_since_last_order
FROM b2b_ecommerce.gold.kpi_6_customer_activity

In [0]:
%sql
-- Most active customers (orders placed), with their company, AOV and recency.
SELECT customer_id, company_name, country, gender, is_active,
       order_count, net_order_count, is_repeat_customer,
       gross_spend, net_spend, avg_order_value_gross, avg_order_value_net,
       first_order_date, last_order_date, days_since_last_order
FROM b2b_ecommerce.gold.kpi_6_customer_activity
ORDER BY order_count DESC, customer_id
LIMIT 10